In [29]:
# !pip install lightgbm xgboost catboost -q

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

pd.set_option("display.max_columns", None)

In [31]:
df = pd.read_csv("/content/model2_volume_predictor.csv")

print(df.shape)

df.head()

(64526, 33)


,h3_cell,date,hour,datetime,violation_count,h3_total_violations,h3_unique_vehicles,h3_hour_density,day_of_week,month,week_of_year,is_weekend,at_junction,at_junction_pct,two_wheeler_pct,car_pct,no_parking_pct,wrong_parking_pct,lag_1h,lag_2h,lag_3h,lag_24h,lag_168h,rolling_3h_mean,rolling_6h_mean,rolling_12h_mean,rolling_24h_mean,count_change_1h,count_change_24h,growth_rate,cell_historical_avg,cell_peak_hour_avg,cell_weekend_avg
0,8860144a65fffff,2024-02-11,8,2024-02-11 08:00:00,5,2,2,2,6,2,6,1,0,0,0.00,0.0,0.4,0.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0
1,8860144b4dfffff,2023-11-14,5,2023-11-14 05:00:00,20,19,2,19,2,11,46,0,0,0,0.95,0.0,0.0,0.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,20.0,20.0
2,8860144b59fffff,2024-02-11,8,2024-02-11 08:00:00,2,1,1,1,6,2,6,1,0,0,0.00,0.0,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0
3,8860144b5bfffff,2024-01-11,3,2024-01-11 03:00:00,1,3,2,1,4,1,2,0,0,0,0.00,0.0,0.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,8860144b5bfffff,2024-01-11,11,2024-01-11 11:00:00,1,3,2,1,3,1,2,0,0,0,0.00,0.0,0.0,1.00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0


In [32]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64526 entries, 0 to 64525
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   h3_cell              64526 non-null  object 
 1   date                 64526 non-null  object 
 2   hour                 64526 non-null  int64  
 3   datetime             64526 non-null  object 
 4   violation_count      64526 non-null  int64  
 5   h3_total_violations  64526 non-null  int64  
 6   h3_unique_vehicles   64526 non-null  int64  
 7   h3_hour_density      64526 non-null  int64  
 8   day_of_week          64526 non-null  int64  
 9   month                64526 non-null  int64  
 10  week_of_year         64526 non-null  int64  
 11  is_weekend           64526 non-null  int64  
 12  at_junction          64526 non-null  int64  
 13  at_junction_pct      64526 non-null  int64  
 14  two_wheeler_pct      64526 non-null  float64
 15  car_pct              64526 non-null 

In [33]:
print(df.isnull().sum().sort_values(ascending=False).head(20))

h3_cell                0
date                   0
hour                   0
datetime               0
violation_count        0
h3_total_violations    0
h3_unique_vehicles     0
h3_hour_density        0
day_of_week            0
month                  0
week_of_year           0
is_weekend             0
at_junction            0
at_junction_pct        0
two_wheeler_pct        0
car_pct                0
no_parking_pct         0
wrong_parking_pct      0
lag_1h                 0
lag_2h                 0
dtype: int64


In [61]:
drop_cols = [
    'date',
    'datetime'
]

for col in drop_cols:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

In [62]:
print(df.columns)


Index(['h3_cell', 'hour', 'violation_count', 'h3_total_violations',
       'h3_unique_vehicles', 'h3_hour_density', 'day_of_week', 'month',
       'week_of_year', 'is_weekend', 'at_junction', 'at_junction_pct',
       'two_wheeler_pct', 'car_pct', 'no_parking_pct', 'wrong_parking_pct',
       'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_168h', 'rolling_3h_mean',
       'rolling_6h_mean', 'rolling_12h_mean', 'rolling_24h_mean',
       'count_change_1h', 'count_change_24h', 'growth_rate',
       'cell_historical_avg', 'cell_peak_hour_avg', 'cell_weekend_avg'],
      dtype='object')


In [63]:
TARGET = 'violation_count'

In [64]:
X = df.drop(TARGET, axis=1)

y = df[TARGET]

In [65]:
cat_cols = X.select_dtypes(
    include=['object']
).columns.tolist()

cat_cols

['h3_cell']

In [66]:
for col in cat_cols:
    X[col] = X[col].astype(str)

In [67]:
split_idx = int(
    len(df) * 0.8
)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

In [68]:
print(X_train.shape)
print(X_test.shape)

(51620, 30)
(12906, 30)


In [69]:
cat_indices = [
    X.columns.get_loc(col)
    for col in cat_cols
]

In [70]:
print(
    y_train.describe()
)

count    51620.000000
mean         5.121038
std          8.571313
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max        356.000000
Name: violation_count, dtype: float64


In [82]:
X_lgb = X.copy()

X_lgb['h3_cell'] = (
    X_lgb['h3_cell']
    .astype('category')
    .cat.codes
)

X_train_lgb = X_lgb.iloc[:split_idx]
X_test_lgb = X_lgb.iloc[split_idx:]


from lightgbm import LGBMRegressor

lgb_model = LGBMRegressor(

    n_estimators=2000,

    learning_rate=0.03,

    num_leaves=63,

    max_depth=10,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)

lgb_model.fit(
    X_train_lgb,
    y_train
)

lgb_preds = lgb_model.predict(
    X_test_lgb
)

print(
    r2_score(
        y_test,
        lgb_preds
    )
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016764 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4075
[LightGBM] [Info] Number of data points in the train set: 51620, number of used features: 30
[LightGBM] [Info] Start training from score 5.121038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [83]:
lgb_preds = lgb_model.predict(
    X_test_lgb
)

lgb_mae = mean_absolute_error(
    y_test,
    lgb_preds
)

lgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        lgb_preds
    )
)

lgb_r2 = r2_score(
    y_test,
    lgb_preds
)

print("LIGHTGBM")

print("MAE :", lgb_mae)
print("RMSE:", lgb_rmse)
print("R2  :", lgb_r2)

LIGHTGBM
MAE : 1.7230901613561032
RMSE: 3.9986559700037247
R2  : 0.8352243071476795


In [44]:
### XGBOOST

xgb_model = XGBRegressor(

    n_estimators=1500,

    learning_rate=0.03,

    max_depth=8,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1500,
             n_jobs=-1, num_parallel_tree=None, ...)

In [45]:
xgb_preds = xgb_model.predict(
    X_test
)

xgb_mae = mean_absolute_error(
    y_test,
    xgb_preds
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_preds
    )
)

xgb_r2 = r2_score(
    y_test,
    xgb_preds
)

print("XGBOOST")

print("MAE :", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2  :", xgb_r2)

XGBOOST
MAE : 2.222121000289917
RMSE: 4.486540268302689
R2  : 0.7925620675086975


In [71]:
### CATBOOST

from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(

    iterations=2000,

    learning_rate=0.03,

    depth=8,

    loss_function='RMSE',

    eval_metric='RMSE',

    random_seed=42,

    verbose=200
)

cat_model.fit(

    X_train,
    y_train,

    cat_features=cat_indices
)

0:	learn: 8.4499477	total: 66.8ms	remaining: 2m 13s
200:	learn: 4.0470860	total: 10.9s	remaining: 1m 37s
400:	learn: 3.4928462	total: 21s	remaining: 1m 23s
600:	learn: 3.1835865	total: 31.5s	remaining: 1m 13s
800:	learn: 3.0081178	total: 46.1s	remaining: 1m 8s
1000:	learn: 2.8727626	total: 57.8s	remaining: 57.7s
1200:	learn: 2.7508439	total: 1m 10s	remaining: 46.6s
1400:	learn: 2.6362940	total: 1m 21s	remaining: 35s
1600:	learn: 2.5444337	total: 1m 33s	remaining: 23.4s
1800:	learn: 2.4628736	total: 1m 45s	remaining: 11.6s
1999:	learn: 2.3944375	total: 1m 59s	remaining: 0us


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=2000, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=200)

In [79]:
preds = cat_model.predict(
    X_test
)

preds = np.clip(preds, 0, None)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

r2 = r2_score(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

mae = mean_absolute_error(
    y_test,
    preds
)

print("R² :", r2)
print("RMSE :", rmse)
print("MAE :", mae)

R² : 0.8421613057139792
RMSE : 3.913579920965879
MAE : 1.5980814147648141


In [80]:
print(preds.min())
print(preds.max())

0.0
127.31325592955434


In [75]:
comparison = pd.DataFrame({

    "Actual": y_test[:20].values,

    "Predicted": preds[:20]

})

comparison

,Actual,Predicted
0,6,6.842042
1,6,3.584504
2,2,2.174522
3,13,14.271962
4,7,7.517731
5,11,10.766136
6,9,9.649101
7,3,3.322991
8,1,1.098170
9,10,9.910569


In [77]:
print(df['violation_count'].describe())

count    64526.000000
mean         5.272479
std          8.847220
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        356.000000
Name: violation_count, dtype: float64


In [49]:
results = pd.DataFrame({

    "Model": [
        "LightGBM",
        "XGBoost",
        "CatBoost"
    ],

    "R2": [
        lgb_r2,
        xgb_r2,
        cat_r2
    ],

    "RMSE": [
        lgb_rmse,
        xgb_rmse,
        cat_rmse
    ],

    "MAE": [
        lgb_mae,
        xgb_mae,
        cat_mae
    ]
})

results.sort_values(
    "R2",
    ascending=False
)

,Model,R2,RMSE,MAE
2,CatBoost,0.837146,3.975274,1.646651
0,LightGBM,0.835448,3.995940,1.696624
1,XGBoost,0.792562,4.486540,2.222121


In [50]:
df.columns.tolist()

['h3_cell',
 'hour',
 'violation_count',
 'h3_total_violations',
 'h3_unique_vehicles',
 'h3_hour_density',
 'day_of_week',
 'month',
 'week_of_year',
 'is_weekend',
 'at_junction',
 'at_junction_pct',
 'two_wheeler_pct',
 'car_pct',
 'no_parking_pct',
 'wrong_parking_pct',
 'lag_1h',
 'lag_2h',
 'lag_3h',
 'lag_24h',
 'lag_168h',
 'rolling_3h_mean',
 'rolling_6h_mean',
 'rolling_12h_mean',
 'rolling_24h_mean',
 'count_change_1h',
 'count_change_24h',
 'growth_rate',
 'cell_historical_avg',
 'cell_peak_hour_avg',
 'cell_weekend_avg']

In [86]:
ensemble_preds = (
    0.7 * preds +
    0.3 * lgb_preds
)

ensemble_preds = np.clip(
    ensemble_preds,
    0,
    None
)

In [87]:
r2_score(
    y_test,
    ensemble_preds
)

0.8484264153246477

In [88]:
joblib.dump(
    cat_model,
    "volume_model.pkl"
)

joblib.dump(
    X.columns.tolist(),
    "volume_features.pkl"
)

['volume_features.pkl']

In [90]:
import json

metrics = {
    "model": "CatBoost",
    "R2": float(0.8420785497599679),
    "RMSE": float(3.9146057454648826),
    "MAE": float(1.6002320906335403)
}

with open("volume_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("volume_metrics.json saved!")

volume_metrics.json saved!


In [91]:
with open("volume_metrics.json", "r") as f:
    print(f.read())

{
    "model": "CatBoost",
    "R2": 0.8420785497599679,
    "RMSE": 3.9146057454648826,
    "MAE": 1.6002320906335403
}


In [ ]:
from google.colab import files

files.download("volume_model.pkl")
files.download("volume_features.pkl")
files.download("volume_metrics.json")

In [92]:
feature_info = {
    "target": "violation_count",
    "model_type": "CatBoostRegressor",
    "input_features": X.columns.tolist()
}

with open("volume_metadata.json", "w") as f:
    json.dump(feature_info, f, indent=4)